# 03 Query Examples

Inspect generated PubMed queries and LLM prompt templates before running data collection.

## Inputs

- Pipeline configuration, pathogen and antibiotic vocabulary tables, and the configured AMR query terms and model prompt templates.

## Outputs

- Rendered PubMed query and prompt examples in the notebook for inspection. No collection or prediction data files are written.


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "amr").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "amr").exists():
    raise RuntimeError(f"Could not find repository root from {Path.cwd()}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from amr.screening import UNIFIED_AMR_INTEREST_SYSTEM_PROMPT
from amr.pipelines import (
    print_pubmed_query_examples,
    print_unified_prompt_examples,
    print_wrapped_system_prompt,
)
from amr.pipeline_config import cfg, cfg_path, load_pipeline_config


## Glossary

- **AMR**: Antimicrobial Resistance.
- **SI**: Scientific Interest score, defined as residual AMR attention after background adjustment.
- **TIAB**: Title/Abstract PubMed field selector (search terms constrained to title or abstract text).
- **PMID**: PubMed Identifier for a publication record.
- **LLM**: Large Language Model used for abstract screening.
- **BG / background counts**: Yearly publication marginals for pathogens or antibiotics used in SI baseline modeling.
- **Slice**: A chosen year interval (`SLICE_START_YEAR` to `SLICE_END_YEAR`) processed in one run.


## Configuration

Keep these parameters aligned with `04_collect_yearly_data.ipynb` before running collection.

In [3]:
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "amr").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
PIPELINE_CONFIG = load_pipeline_config(project_root=PROJECT_ROOT)

ASSETS_DIR = cfg_path(PIPELINE_CONFIG, "paths.assets_dir", PROJECT_ROOT)
OUTPUT_DIR = cfg_path(PIPELINE_CONFIG, "paths.output_dir", PROJECT_ROOT)
PARQUET_OUTPUT_DIR = cfg_path(PIPELINE_CONFIG, "paths.parquet_dir", PROJECT_ROOT)
PARQUET_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = PARQUET_OUTPUT_DIR

PATHOGENS_PATH = cfg_path(PIPELINE_CONFIG, "paths.pathogens", PROJECT_ROOT)
PATHOGEN_PRIORITY_PATH = cfg_path(PIPELINE_CONFIG, "paths.pathogen_priority", PROJECT_ROOT)
MIN_PATHOGENIC_INDEX = cfg(PIPELINE_CONFIG, "collection.min_pathogenic_index")
ANTIBIOTIC_COARSENING_PATH = cfg_path(PIPELINE_CONFIG, "paths.antibiotic_coarsening", PROJECT_ROOT)
ANTIBIOTIC_SYNONYM_PATH = cfg_path(PIPELINE_CONFIG, "paths.antibiotic_synonyms", PROJECT_ROOT)
OUTPUT_YEARLY_AMR_PATH = cfg_path(PIPELINE_CONFIG, "paths.yearly_candidates", PROJECT_ROOT)
OUTPUT_YEARLY_SCREEN_PATH = cfg_path(PIPELINE_CONFIG, "paths.yearly_screened", PROJECT_ROOT)
OUTPUT_UNIFIED_DECISIONS_PATH = cfg_path(PIPELINE_CONFIG, "paths.unified_decisions", PROJECT_ROOT)
OUTPUT_PATHOGEN_BG_PATH = cfg_path(PIPELINE_CONFIG, "paths.pathogen_background", PROJECT_ROOT)
OUTPUT_ANTIBIOTIC_BG_PATH = cfg_path(PIPELINE_CONFIG, "paths.antibiotic_background", PROJECT_ROOT)
INTERVAL_PMIDS_PATH = cfg_path(PIPELINE_CONFIG, "paths.interval_pmids", PROJECT_ROOT)
YEARLY_PAIR_PMID_PATH = cfg_path(PIPELINE_CONFIG, "paths.yearly_pair_pmids", PROJECT_ROOT)
PUBMED_METADATA_CACHE_PATH = cfg_path(PIPELINE_CONFIG, "paths.pubmed_metadata_cache", PROJECT_ROOT)
ABSTRACT_CACHE_PATH = cfg_path(PIPELINE_CONFIG, "paths.abstract_cache", PROJECT_ROOT)
PUBMED_EMAIL = cfg(PIPELINE_CONFIG, "collection.pubmed_email")

# Date windows: TOTAL_* defines the canonical range; SLICE_* limits this run for partitioned/resumable work.
TOTAL_START_YEAR = cfg(PIPELINE_CONFIG, "collection.total_start_year")
TOTAL_END_YEAR = cfg(PIPELINE_CONFIG, "collection.total_end_year")
TOTAL_MIN_DATE = f"{TOTAL_START_YEAR}/01/01"
TOTAL_MAX_DATE = f"{TOTAL_END_YEAR}/12/31"
SLICE_START_YEAR = cfg(PIPELINE_CONFIG, "collection.slice_start_year")
SLICE_END_YEAR = cfg(PIPELINE_CONFIG, "collection.slice_end_year")
MIN_DATE = f"{SLICE_START_YEAR}/01/01"
MAX_DATE = f"{SLICE_END_YEAR}/12/31"
INTERVAL_YEAR_GAP = cfg(PIPELINE_CONFIG, "collection.interval_year_gap")

# Vocabulary limits: use None for full canonical runs; integers are useful for quick/resumable subsets.
NUM_PATHOGENS = cfg(PIPELINE_CONFIG, "collection.num_pathogens")
NUM_ANTIBIOTICS = cfg(PIPELINE_CONFIG, "collection.num_antibiotics")
EXTRA_AMR_KEYWORDS = cfg(PIPELINE_CONFIG, "collection.extra_amr_keywords", [])

# LLM decision modes: rerun_unified_decisions=True ignores active decisions but reuses cached abstracts.
RUN_LLM_FILTER = cfg(PIPELINE_CONFIG, "llm.run_filter")
RERUN_UNIFIED_LLM_DECISIONS = cfg(PIPELINE_CONFIG, "llm.rerun_unified_decisions")
MAX_LLM_CALLS = cfg(PIPELINE_CONFIG, "llm.max_calls")
UNIFIED_SYSTEM_PROMPT = UNIFIED_AMR_INTEREST_SYSTEM_PROMPT
LLM_MAX_WORKERS = cfg(PIPELINE_CONFIG, "llm.max_workers")
LLM_BATCH_SIZE = cfg(PIPELINE_CONFIG, "llm.batch_size")
FAILED_RETRY_PATH = cfg_path(PIPELINE_CONFIG, "paths.unified_failures", PROJECT_ROOT)
MAX_FAILURE_RETRIES = cfg(PIPELINE_CONFIG, "llm.max_failure_retries")
ABSTRACT_FETCH_BATCH_SIZE = cfg(PIPELINE_CONFIG, "llm.abstract_fetch_batch_size")
PREFETCH_ABSTRACTS = cfg(PIPELINE_CONFIG, "llm.prefetch_abstracts")
LLM_CATEGORY_BACKFILL_DECISIONS = cfg(PIPELINE_CONFIG, "llm.category_backfill_decisions")
UNIFIED_PROGRESS_EVERY = cfg(PIPELINE_CONFIG, "llm.progress_every")
SCREENED_INCLUDE_DECISIONS = tuple(cfg(PIPELINE_CONFIG, "llm.screened_include_decisions", ["strict", "yes"]))

# PubMed throughput and background-count controls.
COLLECTION_VERBOSE = cfg(PIPELINE_CONFIG, "collection.collection_verbose")
COLLECTION_PROGRESS_EVERY = cfg(PIPELINE_CONFIG, "collection.collection_progress_every")
PAIR_FULL_FETCH_THRESHOLD = cfg(PIPELINE_CONFIG, "collection.pair_full_fetch_threshold")
BACKGROUND_SCOPE = cfg(PIPELINE_CONFIG, "collection.background_scope")
BACKGROUND_FETCH_THRESHOLD = cfg(PIPELINE_CONFIG, "collection.background_fetch_threshold")
BACKGROUND_METADATA_BATCH_SIZE = cfg(PIPELINE_CONFIG, "collection.background_metadata_batch_size")


## Load Prepared Pathogen And Antibiotic Inputs

In [4]:
bacteria_synonym_df = pd.read_csv(PATHOGENS_PATH)
antibiotic_df = pd.read_csv(ANTIBIOTIC_COARSENING_PATH)
antibiotic_synonym_df = pd.read_csv(ANTIBIOTIC_SYNONYM_PATH)

if not PATHOGEN_PRIORITY_PATH.exists():
    raise FileNotFoundError(
        f"Pathogen priority file not found: {PATHOGEN_PRIORITY_PATH}. "
        "Run notebooks/01_prepare_inputs.ipynb first."
    )

pathogen_priority_df = pd.read_csv(PATHOGEN_PRIORITY_PATH)
required_priority_cols = {"CanonicalSpecies", "pathogenic_index"}
missing_priority_cols = required_priority_cols - set(pathogen_priority_df.columns)
if missing_priority_cols:
    raise KeyError(
        f"{PATHOGEN_PRIORITY_PATH} is missing required columns: {sorted(missing_priority_cols)}"
    )

pathogen_priority_df = pathogen_priority_df.copy()
pathogen_priority_df["CanonicalSpecies"] = pathogen_priority_df["CanonicalSpecies"].astype(str).str.strip()
pathogen_priority_df["pathogenic_index"] = pd.to_numeric(
    pathogen_priority_df["pathogenic_index"],
    errors="coerce",
)
pathogen_priority_df = pathogen_priority_df.dropna(subset=["CanonicalSpecies", "pathogenic_index"])
pathogen_priority_df = pathogen_priority_df.loc[
    pathogen_priority_df["pathogenic_index"] > float(MIN_PATHOGENIC_INDEX)
]
pathogen_priority_df = pathogen_priority_df.sort_values(
    ["pathogenic_index", "CanonicalSpecies"],
    ascending=[False, True],
).drop_duplicates(subset=["CanonicalSpecies"], keep="first")

all_pathogens = pathogen_priority_df["CanonicalSpecies"].tolist()
all_antibiotics = sorted(antibiotic_df["Antibiotic"].dropna().astype(str).str.strip().unique())

pathogens = all_pathogens if NUM_PATHOGENS is None else all_pathogens[: int(NUM_PATHOGENS)]
antibiotics = all_antibiotics if NUM_ANTIBIOTICS is None else all_antibiotics[: int(NUM_ANTIBIOTICS)]

print(f"Pathogen priority source: {PATHOGEN_PRIORITY_PATH}")
print(f"Full synonym lookup vocabulary: {bacteria_synonym_df['bacteria name'].nunique():,}")
print(
    f"Pathogenic-index pathogen vocabulary: {len(all_pathogens):,} "
    f"(MIN_PATHOGENIC_INDEX>{MIN_PATHOGENIC_INDEX})"
)
print(f"Antibiotic coarsening source: {ANTIBIOTIC_COARSENING_PATH}")
print(f"Antibiotic synonym source: {ANTIBIOTIC_SYNONYM_PATH} ({len(antibiotic_synonym_df):,} rows)")
print(f"Full antibiotic vocabulary: {len(all_antibiotics):,}")
print(f"Active pathogens this run: {len(pathogens):,} (NUM_PATHOGENS={NUM_PATHOGENS})")
print(f"Active antibiotics this run: {len(antibiotics):,} (NUM_ANTIBIOTICS={NUM_ANTIBIOTICS})")
print("Top active pathogens by pathogenic_index:")
display(
    pathogen_priority_df[["CanonicalSpecies", "pathogenic_index", "n_causation", "n_articles"]]
    .head(len(pathogens))
    .reset_index(drop=True)
)



Pathogen priority source: /Users/f.costa/Code/Antimicrobial-Resistance/assets/counts_extended2.csv
Full synonym lookup vocabulary: 19,956
Pathogenic-index pathogen vocabulary: 915 (MIN_PATHOGENIC_INDEX>0.0)
Antibiotic coarsening source: /Users/f.costa/Code/Antimicrobial-Resistance/assets/antibiotic_coarsening_lookup.csv
Antibiotic synonym source: /Users/f.costa/Code/Antimicrobial-Resistance/assets/antibiotic_synonym_lookup.csv (422 rows)
Full antibiotic vocabulary: 245
Active pathogens this run: 200 (NUM_PATHOGENS=200)
Active antibiotics this run: 245 (NUM_ANTIBIOTICS=None)
Top active pathogens by pathogenic_index:


,CanonicalSpecies,pathogenic_index,n_causation,n_articles
0,Salmonella enterica,1.000000,554,24694
1,Mycobacterium tuberculosis,0.928144,299,20559
2,Clostridioides difficile,0.910833,298,17012
3,Haemophilus influenzae,0.860892,200,14537
4,Klebsiella pneumoniae,0.857160,176,15847
...,...,...,...,...
195,Lactococcus lactis,0.369238,11,1109
196,Streptococcus gordonii,0.368655,20,606
197,Aeromonas caviae,0.366312,48,246
198,Pseudomonas oryzihabitans,0.366251,59,94


## PubMed Query Logic for Abstract Collection

The collection query is assembled per `(pathogen, antibiotic)` pair with four components:

1. **Antibiotic clause**: antibiotic name in `Abstract`, including split variants for hyphen/slash names (for example `co-trimoxazole` as exact phrase OR split terms).
2. **Pathogen clause**: pathogen + known synonyms in `Abstract` (`OR`-joined).
3. **AMR clause**: additional AMR-focused condition from `EXTRA_AMR_KEYWORDS` in `Title/Abstract`.
4. **Date clause**: publication date restricted to `[MIN_DATE, MAX_DATE]`.

Final query shape:

`(antibiotic_clause AND pathogen_clause) AND extra_amr_clause AND date_range_clause`


In [5]:
print_pubmed_query_examples(
    pathogens=pathogens,
    antibiotics=antibiotics,
    bacteria_synonym_df=bacteria_synonym_df,
    antibiotic_synonym_df=antibiotic_synonym_df,
    extra_clauses=EXTRA_AMR_KEYWORDS,
    min_date=MIN_DATE,
    max_date=MAX_DATE,
    max_examples=3,
)

Sample pathogen: Salmonella enterica
Sample antibiotic: Amikacin

Sample PubMed query:

(("Amikacin"[Abstract]) AND ("Salmonella enterica"[Abstract] OR "salmonella arizonae"[Abstract] OR "salmonella choleraesuis"[Abstract] OR "salmonella enterica"[Abstract] OR "salmonella enteritidis"[Abstract] OR "salmonella paratyphi"[Abstract] OR "salmonella typhi"[Abstract] OR "salmonella typhimurium"[Abstract])) AND ("antimicrobial resistance"[Title/Abstract] OR "antibiotic resistance"[Title/Abstract]) AND ("2000/01/01"[Date - Publication] : "2026/12/31"[Date - Publication])


EXAMPLE PUBMED QUERIES (MULTIPLE VIEWS)

Example 1: Salmonella enterica vs Amikacin
- Pathogen TIAB clause:
("Salmonella enterica"[Title/Abstract] OR "salmonella arizonae"[Title/Abstract] OR "salmonella choleraesuis"[Title/Abstract] OR "salmonella enterica"[Title/Abstract] OR "salmonella enteritidis"[Title/Abstract] OR "salmonella paratyphi"[Title/Abstract] OR "salmonella typhi"[Title/Abstract] OR "salmonella typhimurium"[Ti

## Resistance-Only LLM Prompt Templates
The classifier screen is restricted to explicit resistance findings for the exact pathogen–antibiotic pair. It no longer assigns AMR-context categories.

**Prompt governance note:** document prompt changes so downstream results remain interpretable.

In [ ]:
if "strict | yes | no | uncertain" not in UNIFIED_SYSTEM_PROMPT:
    raise RuntimeError("UNIFIED_SYSTEM_PROMPT is stale. Restart the kernel or rerun the import/config cells.")

print_unified_prompt_examples(UNIFIED_SYSTEM_PROMPT)
